# Weighted Late Fusion — Task 2 (All Tasks, LOSO)

Combines CNN + ST-GCN + BiLSTM probability outputs using weighted late fusion.

**Before running:**
- Copy `cnn_probs.csv` from Machine 1
- Copy `bilstm_probs.csv` from Machine 2
- Copy `stgcn_probs.csv` from Daniel's machine
- Put all 3 files in `C:\Users\hp\StressDetection\DL\fusion_probs\`

In [1]:
import os
import numpy as np
import pandas as pd
from scipy.optimize import minimize
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    precision_score,
    recall_score,
    confusion_matrix
)

# ------------------------------------------------------------
# Paths
# ------------------------------------------------------------
DL_DIR    = r"C:\Users\hp\StressDetection\DL"
SAVE_DIR  = DL_DIR

CNN_PROBS_PATH    = os.path.join(DL_DIR, "cnn_probs.csv")
BILSTM_PROBS_PATH = os.path.join(DL_DIR, "bilstm_probs.csv")
STGCN_PROBS_PATH  = os.path.join(DL_DIR, "stgcn_probs.csv")

print("Loading probability files...")

cnn_df    = pd.read_csv(CNN_PROBS_PATH)
bilstm_df = pd.read_csv(BILSTM_PROBS_PATH)
stgcn_df  = pd.read_csv(STGCN_PROBS_PATH)

print(f"CNN    probs shape : {cnn_df.shape}")
print(f"BiLSTM probs shape : {bilstm_df.shape}")
print(f"ST-GCN probs shape : {stgcn_df.shape}")
print("\nCNN head:")
print(cnn_df.head())

Loading probability files...
CNN    probs shape : (29735, 3)
BiLSTM probs shape : (29735, 3)
ST-GCN probs shape : (29735, 3)

CNN head:
   pid  true_label  prob_stress
0    1           0     0.910990
1    1           0     0.948127
2    1           0     0.820668
3    1           0     0.976522
4    1           0     0.970606


In [2]:
print("CNN label distribution:")
print(cnn_df.groupby("true_label")["prob_stress"].mean())

print("\nBiLSTM label distribution:")
print(bilstm_df.groupby("true_label")["prob_stress"].mean())

print("\nST-GCN label distribution:")
print(stgcn_df.groupby("true_label")["prob_stress"].mean())

CNN label distribution:
true_label
0    0.152416
1    0.876885
Name: prob_stress, dtype: float64

BiLSTM label distribution:
true_label
0    0.420799
1    0.616119
Name: prob_stress, dtype: float64

ST-GCN label distribution:
true_label
0    0.415240
1    0.837539
Name: prob_stress, dtype: float64


In [3]:
# ------------------------------------------------------------
# Align all 3 CSVs by pid and window order
# ------------------------------------------------------------

# Add window index within each participant
cnn_df    = cnn_df.copy()
bilstm_df = bilstm_df.copy()
stgcn_df  = stgcn_df.copy()

for df in [cnn_df, bilstm_df, stgcn_df]:
    df["win_idx"] = df.groupby("pid").cumcount()

# Rename prob columns before merge
cnn_df    = cnn_df.rename(columns={"prob_stress": "prob_cnn"})
bilstm_df = bilstm_df.rename(columns={"prob_stress": "prob_bilstm"})
stgcn_df  = stgcn_df.rename(columns={"prob_stress": "prob_stgcn"})

# Merge on pid + win_idx
merged = cnn_df[["pid", "win_idx", "true_label", "prob_cnn"]].merge(
    bilstm_df[["pid", "win_idx", "prob_bilstm"]], on=["pid", "win_idx"]
).merge(
    stgcn_df[["pid", "win_idx", "prob_stgcn"]], on=["pid", "win_idx"]
)

print(f"Merged shape : {merged.shape}")
print(f"Participants : {merged['pid'].nunique()}")
print(f"Missing values: {merged.isnull().sum().sum()}")
print()
print(merged.head(10))

Merged shape : (29735, 6)
Participants : 56
Missing values: 0

   pid  win_idx  true_label  prob_cnn  prob_bilstm  prob_stgcn
0    1        0           0  0.910990     0.357739    0.989549
1    1        1           0  0.948127     0.348952    0.961940
2    1        2           0  0.820668     0.371834    0.969341
3    1        3           0  0.976522     0.329531    0.973930
4    1        4           0  0.970606     0.334825    0.982311
5    1        5           0  0.912416     0.328630    0.973130
6    1        6           0  0.872614     0.831663    0.970380
7    1        7           0  0.492368     0.815703    0.978261
8    1        8           0  0.458340     0.832761    0.991848
9    1        9           0  0.498718     0.834973    0.996197


In [4]:
# ------------------------------------------------------------
# Weighted Late Fusion — find optimal weights
# ------------------------------------------------------------
# Strategy: LOSO — for each test participant, optimize weights
# on the remaining 55 participants (train set), then evaluate on test.
# This avoids any leakage.
# ------------------------------------------------------------

def fuse(probs_cnn, probs_bilstm, probs_stgcn, weights):
    """Weighted combination of 3 probability streams."""
    w_cnn, w_bilstm, w_stgcn = weights
    return w_cnn * probs_cnn + w_bilstm * probs_bilstm + w_stgcn * probs_stgcn


def optimize_weights(train_df):
    """Find weights that maximize balanced accuracy on train set."""
    def neg_bal_acc(w):
        # Softmax to ensure weights sum to 1 and are positive
        w_soft = np.exp(w) / np.exp(w).sum()
        fused  = fuse(
            train_df["prob_cnn"].values,
            train_df["prob_bilstm"].values,
            train_df["prob_stgcn"].values,
            w_soft
        )
        preds = (fused >= 0.5).astype(int)
        return -balanced_accuracy_score(train_df["true_label"].values, preds)

    # Start from equal weights
    w0     = np.array([1.0, 1.0, 1.0])
    result = minimize(neg_bal_acc, w0, method="Nelder-Mead",
                      options={"maxiter": 1000, "xatol": 1e-4})
    best_w = np.exp(result.x) / np.exp(result.x).sum()
    return best_w


print("Weight optimization function defined.")

Weight optimization function defined.


In [5]:
# ------------------------------------------------------------
# Full LOSO Fusion Evaluation
# ------------------------------------------------------------

fusion_results = []
unique_pids    = sorted(merged["pid"].unique())

print(f"Running LOSO fusion on {len(unique_pids)} participants...")
print()

for fold_i, test_pid in enumerate(unique_pids, start=1):

    test_df  = merged[merged["pid"] == test_pid]
    train_df = merged[merged["pid"] != test_pid]

    # Optimize weights on training participants
    best_weights = optimize_weights(train_df)

    # Apply to test participant
    fused_probs = fuse(
        test_df["prob_cnn"].values,
        test_df["prob_bilstm"].values,
        test_df["prob_stgcn"].values,
        best_weights
    )

    y_true = test_df["true_label"].values
    y_pred = (fused_probs >= 0.5).astype(int)

    fold_result = {
        "pid":          test_pid,
        "w_cnn":        round(best_weights[0], 4),
        "w_bilstm":     round(best_weights[1], 4),
        "w_stgcn":      round(best_weights[2], 4),
        "accuracy":     accuracy_score(y_true, y_pred),
        "balanced_acc": balanced_accuracy_score(y_true, y_pred),
        "f1":           f1_score(y_true, y_pred, zero_division=0),
        "precision":    precision_score(y_true, y_pred, zero_division=0),
        "recall":       recall_score(y_true, y_pred, zero_division=0),
        "n_windows":    len(y_true)
    }

    fusion_results.append(fold_result)

    print(
        f"[{fold_i:2d}/{len(unique_pids)}] p{test_pid:2d} | "
        f"w=[CNN:{best_weights[0]:.2f} BiLSTM:{best_weights[1]:.2f} ST-GCN:{best_weights[2]:.2f}] | "
        f"Bal.Acc={fold_result['balanced_acc']:.4f} | "
        f"F1={fold_result['f1']:.4f}"
    )

print()
print("LOSO fusion complete!")

Running LOSO fusion on 56 participants...

[ 1/56] p 1 | w=[CNN:0.64 BiLSTM:0.24 ST-GCN:0.12] | Bal.Acc=0.6737 | F1=0.8571
[ 2/56] p 2 | w=[CNN:0.62 BiLSTM:0.31 ST-GCN:0.07] | Bal.Acc=0.9506 | F1=0.9564
[ 3/56] p 3 | w=[CNN:0.62 BiLSTM:0.31 ST-GCN:0.07] | Bal.Acc=0.9138 | F1=0.9453
[ 4/56] p 4 | w=[CNN:0.63 BiLSTM:0.30 ST-GCN:0.06] | Bal.Acc=0.9562 | F1=0.9672
[ 5/56] p 5 | w=[CNN:0.63 BiLSTM:0.26 ST-GCN:0.11] | Bal.Acc=0.9605 | F1=0.9685
[ 6/56] p 6 | w=[CNN:0.63 BiLSTM:0.26 ST-GCN:0.12] | Bal.Acc=0.8771 | F1=0.8599
[ 7/56] p 7 | w=[CNN:0.64 BiLSTM:0.24 ST-GCN:0.12] | Bal.Acc=0.9096 | F1=0.9441
[ 8/56] p 8 | w=[CNN:0.64 BiLSTM:0.24 ST-GCN:0.12] | Bal.Acc=0.9463 | F1=0.9694
[ 9/56] p 9 | w=[CNN:0.62 BiLSTM:0.31 ST-GCN:0.07] | Bal.Acc=0.8051 | F1=0.7894
[10/56] p10 | w=[CNN:0.63 BiLSTM:0.25 ST-GCN:0.12] | Bal.Acc=0.9294 | F1=0.9644
[11/56] p11 | w=[CNN:0.63 BiLSTM:0.26 ST-GCN:0.11] | Bal.Acc=0.9379 | F1=0.9638
[12/56] p12 | w=[CNN:0.64 BiLSTM:0.30 ST-GCN:0.06] | Bal.Acc=0.9958 | F1=0.99

In [6]:
# ------------------------------------------------------------
# Final Results Summary
# ------------------------------------------------------------

results_df = pd.DataFrame(fusion_results)

print("=" * 65)
print("WEIGHTED LATE FUSION — LOSO RESULTS ")
print("=" * 65)
print(f"Folds completed  : {len(results_df)}")
print(f"Mean Accuracy    : {results_df['accuracy'].mean()*100:.2f}%")
print(f"Mean Balanced Acc: {results_df['balanced_acc'].mean()*100:.2f}%")
print(f"Mean F1          : {results_df['f1'].mean():.4f}")
print(f"Mean Precision   : {results_df['precision'].mean():.4f}")
print(f"Mean Recall      : {results_df['recall'].mean():.4f}")
print()
print("Mean fusion weights:")
print(f"  CNN    : {results_df['w_cnn'].mean():.4f}")
print(f"  BiLSTM : {results_df['w_bilstm'].mean():.4f}")
print(f"  ST-GCN : {results_df['w_stgcn'].mean():.4f}")
print()
print("=" * 65)
print(f"ML Baseline (XGBoost LOSO) : 92.70%")
print(f"CNN alone                  : 90.18%")
print(f"ST-GCN alone               : 75.70%")
print(f"BiLSTM alone               : 70.00%")
print(f"Fusion (this)              : {results_df['balanced_acc'].mean()*100:.2f}%")
print("=" * 65)

# Save results
save_path = os.path.join(SAVE_DIR, "fusion_weighted_late_results.csv")
results_df.to_csv(save_path, index=False)
print(f"\nResults saved → {save_path}")
results_df

WEIGHTED LATE FUSION — LOSO RESULTS 
Folds completed  : 56
Mean Accuracy    : 90.92%
Mean Balanced Acc: 90.77%
Mean F1          : 0.9273
Mean Precision   : 0.9535
Mean Recall      : 0.9120

Mean fusion weights:
  CNN    : 0.6271
  BiLSTM : 0.2872
  ST-GCN : 0.0857

ML Baseline (XGBoost LOSO) : 92.70%
CNN alone                  : 90.18%
ST-GCN alone               : 75.70%
BiLSTM alone               : 70.00%
Fusion (this)              : 90.77%

Results saved → C:\Users\hp\StressDetection\DL\fusion_weighted_late_results.csv


,pid,w_cnn,w_bilstm,w_stgcn,accuracy,balanced_acc,f1,precision,recall,n_windows
0,1,0.6362,0.2443,0.1195,0.779661,0.673729,0.857143,0.754839,0.991525,531
1,2,0.6201,0.3088,0.0711,0.943503,0.950565,0.956395,0.985030,0.929379,531
2,3,0.6228,0.3054,0.0718,0.926554,0.913842,0.945302,0.938719,0.951977,531
3,4,0.6343,0.3015,0.0642,0.956685,0.956215,0.967190,0.976945,0.957627,531
4,5,0.6276,0.2615,0.1110,0.958569,0.960452,0.968481,0.982558,0.954802,531
5,6,0.6271,0.2561,0.1168,0.836158,0.877119,0.859903,1.000000,0.754237,531
6,7,0.6379,0.2427,0.1194,0.924670,0.909605,0.944134,0.933702,0.954802,531
7,8,0.6370,0.2436,0.1193,0.958569,0.946328,0.969359,0.956044,0.983051,531
8,9,0.6233,0.3055,0.0712,0.760829,0.805085,0.789386,0.955823,0.672316,531
9,10,0.6323,0.2508,0.1169,0.951036,0.929379,0.964384,0.936170,0.994350,531
